# Multi Agent Systems & Workflow Patterns

In [1]:
import os

## Verify and Authenticate the ADK

In [2]:
def verify_and_authenticate():
    api_key = os.environ.get("GOOGLE_API_KEY")
    if api_key:
        print("Key is configured")
        return
    print("Key not found")
    return 

In [3]:
verify_and_authenticate()

Key is configured


## Importing ADK Components

In [5]:
from google.adk.agents import Agent, SequentialAgent, ParallelAgent, LoopAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools import AgentTool, FunctionTool, google_search
from google.genai import types

print("ADK Components are imported successfully")

ADK Components are imported successfully


## Configure retry options

In [7]:
retry_config = types.HttpRetryOptions(attempts=5,expBase=7,initialDelay=1,httpStatusCodes=[429,500,503,504])

## Example 1: Reasearch & Summarization Agent

Let's build a system with two specialized agents:

- *Research Agent* - Searches for information using Google Search
- *Summarizer Agent* - Creates concise summaries from research findings

### Reasearch Agent

In [9]:
research_agent = Agent(
    name = "ResearchAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction=""" You are a specialized research agent. Your only job is to use the 
    google_search tool to find 2-3 pieces of relevant information on the given topic and present
    the findings with citations""",
    output_key="research_findings"
)
print("Research Agent Created")

Research Agent Created


### Summarizer Agent

In [10]:
summarizer_agent = Agent(
    name="SummarizerAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction=""" Read the provided research findings: {research_findings}
    Create a concise summary as a bulleted list with 3-5 key points
    """,
    output_key="final_summary"
)
print("summarizer_agent created")

summarizer_agent created


### Research Coordinator Agent

Orchestrates the workflow by calling the sub-agents as tools

In [12]:
root_agent = Agent(
    name="ResearchCoordinator",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction=""" You are a research coordinator. Your goal is to answer the user's query by orchestrating a workflow.
    1. First, you MUST call the `ResearchAgent` tool to find the relavent information on the topic provided by the user.
    2. Next, after receiving the research findings, you MUST call the `SummarizerAgent` tool to create a concise summary.
    3. Finally, present the final summary clearly to the user as your response.""",
    tools=[AgentTool(research_agent), AgentTool(summarizer_agent)]
)

print("root_agent created")

root_agent created


## Configuring the runner

In [15]:
runner = InMemoryRunner(agent=root_agent)

In [16]:
await runner.run_debug(
    "What are the latest advancements in quantum computing and what do they mean for AI?"
)


 ### Created new session: debug_session_id

User > What are the latest advancements in quantum computing and what do they mean for AI?
ResearchCoordinator > Here's a concise summary of the latest advancements in quantum computing and their implications for AI:

*   **Improved Quantum Hardware:** Significant progress is being made in qubit stability and connectivity, enabling more complex quantum computations essential for advanced AI.
*   **Specialized Quantum Algorithms for AI:** Researchers are developing quantum algorithms, particularly in quantum machine learning, to accelerate and enhance AI tasks like pattern recognition and optimization.
*   **Hybrid Quantum-Classical Systems:** Combining quantum and classical computing offers a practical approach to leveraging quantum advantage for AI in the near term by offloading intensive subroutines to quantum processors.
*   **Revolutionary AI Potential:** These advancements promise to lead to faster, more efficient AI models, unlock nove

[Event(model_version='gemini-2.5-flash-lite', content=Content(
   parts=[
     Part(
       function_call=FunctionCall(
         args={
           'request': 'latest advancements in quantum computing and their implications for AI'
         },
         id='adk-c4d1823f-d754-42b6-b384-e5e204c7e2e9',
         name='ResearchAgent'
       )
     ),
   ],
   role='model'
 ), grounding_metadata=None, partial=None, turn_complete=None, finish_reason=<FinishReason.STOP: 'STOP'>, error_code=None, error_message=None, interrupted=None, custom_metadata=None, usage_metadata=GenerateContentResponseUsageMetadata(
   candidates_token_count=23,
   prompt_token_count=192,
   prompt_tokens_details=[
     ModalityTokenCount(
       modality=<MediaModality.TEXT: 'TEXT'>,
       token_count=192
     ),
   ],
   total_token_count=215
 ), live_session_resumption_update=None, input_transcription=None, output_transcription=None, avg_logprobs=None, logprobs_result=None, cache_metadata=None, citation_metadata=None,

## Sequential Workflows - The Assembly Line

### Example Blog Post Creation with Sequential Agents

Let's build a system with three specialized agents:
1. **Outline Agent** - Creates a blog outline for a given topic.
2. **Writer Agent** - Writes a blog post
3. **Editor Agent** - Edits a blog post draft for clarity and structure

#### Outline Agent: Creates the initial blog post outline

In [24]:
outline_agent = Agent(
    name="OutlineAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction=""" Create a blog outline for the given topic with:
    1. A catchy headline.
    2. An Intruduction hook
    3. 3-5 main sections with 2-3 bullet points for each
    4. A concluding thought """,
    output_key="blog_outline"
)
print("outline_agent created")

outline_agent created


#### Writer Agent: Edits and polishes the draft from the writer agent

In [25]:
writer_agent = Agent(
    name="WriterAgent",
    model=Gemini(
        model="gemini-2.5-flash-lite",
        retry_options=retry_config
    ),
    instruction=""" Following this outline strictly: {blog_outline}
    write a brief, 200 to 300-word blog post with an engaging and informative tone.""",
    output_key="blog_draft"
)

print("writer_agent created")

writer_agent created


#### Editor Agent: Edits and ploshes the fraft from the writer agent

In [26]:
editor_agent = Agent(
    name="EditorAgent",
    model=Gemini(model="gemini-2.5-flash-lite", retry_options=retry_config),
    instruction="""Edit this draft: {blog_draft}
    Your task is to polish the test by fixing any grammatical errors, improving the flow and sentence structure, and enhancing overall clarity.""",
    output_key="final_blog"
)
print("editor agent created")

editor agent created


#### Bringing Agents together By using the sequential agent tool

In [27]:
seq_agent = SequentialAgent(
    name="BlogPipeline",
    sub_agents=[outline_agent, writer_agent, editor_agent]
)
print("Pipeline created")

Pipeline created


#### Configuring the runner for blog post creator

In [29]:
seq_runner = InMemoryRunner(agent=seq_agent)

In [31]:
await seq_runner.run_debug(
    """Write a blog post about the benifits of multi-agent systems for the software developers, include outline and basic syntax to create a multi-agent systems"""
)


 ### Created new session: debug_session_id

User > Write a blog post about the benifits of multi-agent systems for the software developers, include outline and basic syntax to create a multi-agent systems
OutlineAgent > ## Unleash Your Development Superpowers: The Multi-Agent System Advantage

**Introduction Hook:** Imagine a team of tireless, specialized assistants working collaboratively on your software projects, each an expert in its own domain, communicating seamlessly, and driving towards a common goal. This isn't science fiction; it's the power of multi-agent systems (MAS) for software developers, and it's about to revolutionize how you build and deploy applications.

### Section 1: The Developer's Dilemma – Complexity and Collaboration

*   **The Ever-Growing Software Landscape:** Modern software is becoming increasingly complex, demanding sophisticated solutions and efficient development processes. Traditional monolithic approaches can lead to bottlenecks and unmanageable cod

[Event(model_version='gemini-2.5-flash-lite', content=Content(
   parts=[
     Part(
       text="""## Unleash Your Development Superpowers: The Multi-Agent System Advantage
 
 **Introduction Hook:** Imagine a team of tireless, specialized assistants working collaboratively on your software projects, each an expert in its own domain, communicating seamlessly, and driving towards a common goal. This isn't science fiction; it's the power of multi-agent systems (MAS) for software developers, and it's about to revolutionize how you build and deploy applications.
 
 ### Section 1: The Developer's Dilemma – Complexity and Collaboration
 
 *   **The Ever-Growing Software Landscape:** Modern software is becoming increasingly complex, demanding sophisticated solutions and efficient development processes. Traditional monolithic approaches can lead to bottlenecks and unmanageable codebases.
 *   **The Need for Specialization and Autonomy:** Developers often specialize, but coordinating these spec